# Reclassify: Human=diagram but CLIP disagrees
Navigate through all 40 figures. Click a category button to reclassify, or **Skip** to keep as diagram.

In [ ]:
import json
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

FIG_RAG_DIR = Path(os.environ.get('FIG_RAG_DIR', '~/fig-rag-dir')).expanduser()
A1 = Path.cwd()  # run from experiments/clip_router_audit
ANNOT_PATH = A1 / 'annotations_tingwen.jsonl'

# Load human annotations
annots = []
with open(ANNOT_PATH) as f:
    for line in f:
        annots.append(json.loads(line))
annot_lookup = {}
for i, rec in enumerate(annots):
    key = (rec['venue'], rec['platform_id'], rec['figure_number'])
    annot_lookup[key] = i

# Load sample
sample = {}
with open(A1 / 'sample.jsonl') as f:
    for line in f:
        rec = json.loads(line)
        key = (rec['venue'], rec['platform_id'], rec['figure_number'])
        sample[key] = rec

# Find mismatches
mismatches = []
for rec in annots:
    key = (rec['venue'], rec['platform_id'], rec['figure_number'])
    label = rec.get('label_tingwen', '').strip().lower()
    if label == 'diagram' and key in sample:
        s = sample[key]
        if s['figure_type'] != 'diagram':
            mismatches.append({
                'key': key,
                'venue': s['venue'],
                'platform_id': s['platform_id'],
                'figure_number': s['figure_number'],
                'clip_type': s['figure_type'],
                'confidence': s['confidence'],
                'figure_path': s['figure_path'],
            })
mismatches.sort(key=lambda x: (x['clip_type'], -x['confidence']))
print(f'Loaded {len(mismatches)} figures to review')

In [ ]:
# State
idx = [0]
changes = {}

# Widgets
img_widget = widgets.Image(layout=widgets.Layout(max_width='700px', max_height='600px'))
info_label = widgets.HTML()
status_label = widgets.HTML()
changes_label = widgets.HTML()

btn_diagram = widgets.Button(description='diagram', button_style='info',
                              layout=widgets.Layout(width='120px', height='40px'))
btn_plot = widgets.Button(description='plot', button_style='warning',
                           layout=widgets.Layout(width='120px', height='40px'))
btn_photo = widgets.Button(description='photo', button_style='success',
                            layout=widgets.Layout(width='120px', height='40px'))
btn_other = widgets.Button(description='other', button_style='danger',
                            layout=widgets.Layout(width='120px', height='40px'))
btn_prev = widgets.Button(description='< Prev', layout=widgets.Layout(width='100px', height='40px'))
btn_next = widgets.Button(description='Next >', layout=widgets.Layout(width='100px', height='40px'))
btn_save = widgets.Button(description='Save All', button_style='success',
                           layout=widgets.Layout(width='140px', height='45px'))
jump_input = widgets.BoundedIntText(value=0, min=0, max=len(mismatches)-1,
                                     description='Go to #:', layout=widgets.Layout(width='180px'))

def show_figure():
    i = idx[0]
    m = mismatches[i]
    fig_path = FIG_RAG_DIR / m['figure_path']
    
    if fig_path.exists():
        img_widget.value = fig_path.read_bytes()
        img_widget.format = 'png'
    else:
        img_widget.value = b''
    
    current_label = changes.get(m['key'], 'diagram')
    changed_marker = ' <span style="color:orange; font-weight:bold">[CHANGED]</span>' if m['key'] in changes else ''
    
    info_label.value = (
        f'<div style="font-size:16px; padding:8px; background:#f0f0f0; border-radius:5px;">'
        f'<b>#{i}/{len(mismatches)-1}</b> &nbsp; '
        f'<b>{m["venue"]}</b> {m["platform_id"]} Fig{m["figure_number"]}<br>'
        f'CLIP: <span style="color:red; font-weight:bold">{m["clip_type"]}</span> '
        f'(conf: {m["confidence"]:.3f}) &nbsp; | &nbsp; '
        f'Current label: <span style="color:blue; font-weight:bold">{current_label}</span>'
        f'{changed_marker}</div>'
    )
    
    n_changes = len(changes)
    changes_label.value = (
        f'<div style="font-size:13px; padding:4px;">'
        f'Changes pending: <b>{n_changes}</b></div>'
    )

def classify(label):
    m = mismatches[idx[0]]
    if label == 'diagram':
        changes.pop(m['key'], None)
    else:
        changes[m['key']] = label
    status_label.value = f'<span style="color:green">Set to <b>{label}</b></span>'
    # Auto-advance
    if idx[0] < len(mismatches) - 1:
        idx[0] += 1
    show_figure()

def on_diagram(b): classify('diagram')
def on_plot(b): classify('plot')
def on_photo(b): classify('photo')
def on_other(b): classify('other')

def on_prev(b):
    if idx[0] > 0:
        idx[0] -= 1
        status_label.value = ''
        show_figure()

def on_next(b):
    if idx[0] < len(mismatches) - 1:
        idx[0] += 1
        status_label.value = ''
        show_figure()

def on_jump(change):
    idx[0] = change['new']
    status_label.value = ''
    show_figure()

def on_save(b):
    if not changes:
        status_label.value = '<span style="color:gray">No changes to save.</span>'
        return
    for key, new_label in changes.items():
        annot_idx = annot_lookup[key]
        annots[annot_idx]['label_tingwen'] = new_label
    with open(ANNOT_PATH, 'w') as f:
        for rec in annots:
            f.write(json.dumps(rec) + '\n')
    status_label.value = (
        f'<span style="color:green; font-weight:bold">'
        f'Saved {len(changes)} corrections to annotations_tingwen.jsonl. '
        f'Re-run evaluate.py to refresh results.</span>'
    )
    changes.clear()
    show_figure()

btn_diagram.on_click(on_diagram)
btn_plot.on_click(on_plot)
btn_photo.on_click(on_photo)
btn_other.on_click(on_other)
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
btn_save.on_click(on_save)
jump_input.observe(on_jump, names='value')

# Layout
classify_row = widgets.HBox([btn_diagram, btn_plot, btn_photo, btn_other],
                             layout=widgets.Layout(gap='10px'))
nav_row = widgets.HBox([btn_prev, jump_input, btn_next],
                        layout=widgets.Layout(gap='10px', align_items='center'))
save_row = widgets.HBox([btn_save, changes_label, status_label],
                         layout=widgets.Layout(gap='15px', align_items='center'))

ui = widgets.VBox([
    info_label,
    img_widget,
    widgets.HTML('<b>Classify as:</b>'),
    classify_row,
    nav_row,
    save_row,
], layout=widgets.Layout(gap='8px'))

show_figure()
display(ui)